# EDA 04: Macro, flows and news

- **Period:** 2016-01-01 to 2023-12-31 (train 2016–2022, validation 2023).
- **Time rule:** every macro value is aligned on `known_from`. On trading day d the forecast (08:30 IST) sees
  the latest value with `known_from <= 08:30 IST on d`. FRED values are US market closes, usable from 05:30 IST
  the next day, so day d sees the US close of d−1 (or earlier after a US holiday or weekend). Its change from
  the previous trading day's value is the **overnight US move**. Days with no new US observation get no change.
- **Nifty moves:** log returns from closes (never the file's `change_pct`, wrong on 13 Mar 2023, E01);
  gap = open(d) vs close(d−1); intraday = close(d) vs open(d).
- **Intervals:** 95% moving-block bootstrap (20-day blocks), since volatility clusters and 5-day windows overlap.
- **Helpers:** `src/analysis/macro.py` (`macro_asof`, with a leakage test), `src/features/regime.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis import macro as M
from src.analysis import returns as R
from src.analysis.eda_plots import heatmap, save
from src.config import Config
from src.features import regime as G

pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 90, "display.precision", 4)
cfg = Config.load()
START, TRAIN_END, END = "2016-01-01", "2022-12-31", "2023-12-31"
SEED, NBOOT = 7, 2000
con = duckdb.connect(str(cfg.duckdb_path), read_only=True)
con.execute("SET TimeZone = 'UTC'")

macro_all = con.execute("SELECT series_id, description, obs_date, value, known_from, published FROM macro").df()
macro_all["obs_date"] = pd.to_datetime(macro_all.obs_date)
idx = con.execute("SELECT index_name, trade_date, open, close FROM index_daily WHERE trade_date BETWEEN ? AND ?", [START, END]).df()
NIFTY = idx[idx.index_name == "Nifty 50"].set_index(pd.to_datetime(idx[idx.index_name == "Nifty 50"].trade_date)).sort_index()
IVIX = idx[idx.index_name == "India VIX"].set_index(pd.to_datetime(idx[idx.index_name == "India VIX"].trade_date)).sort_index().close
DAYS = NIFTY.index

SERIES = {"DGS10": "US 10y yield (pp)", "VIXCLS": "US VIX", "DEXINUS": "USD/INR", "DTWEXBGS": "Broad dollar index",
          "DCOILBRENTEU": "Brent crude ($)"}
LOG = ("VIXCLS", "DEXINUS", "DTWEXBGS", "DCOILBRENTEU")          # log changes; DGS10 in percentage points
macro = macro_all[(macro_all.obs_date >= "2015-12-01") & (macro_all.obs_date <= END)]
vals, obs = M.macro_asof(macro, DAYS)
chg = M.overnight_change(vals, obs, LOG)
print(f"{len(DAYS)} NSE trading days; macro rows 2016-2023: {int((macro.obs_date >= START).sum()):,}")

## 1. FRED series: levels, missing days, publication

In [ ]:
m16 = macro_all[(macro_all.obs_date >= START) & (macro_all.obs_date <= END)]
lag = (m16.known_from - m16.obs_date.dt.tz_localize("UTC"))
summary = m16.groupby("series_id").agg(description=("description", "first"), published=("published", "first"),
                                       rows=("value", "size"), missing_values=("value", lambda v: int(v.isna().sum())))
summary["missing share"] = summary.missing_values / summary.rows
summary["known_from - obs_date"] = lag.groupby(m16.series_id).agg(lambda s: ", ".join(sorted(set(str(x) for x in s))))
age = pd.DataFrame({c: (pd.Series(DAYS, index=DAYS) - obs[c]).dt.days for c in obs.columns})
summary["median age on NSE day (days)"] = age.median()
summary["share of NSE days with a new value"] = chg.notna().mean()
summary["max age (days)"] = age.max()
display(summary)
print("missing values by year:")
display(m16.assign(year=m16.obs_date.dt.year).groupby(["series_id", "year"]).value.apply(lambda v: int(v.isna().sum())).unstack())

fig, axes = plt.subplots(len(SERIES), 1, figsize=(13, 11), sharex=True)
for ax, (sid, name) in zip(axes, SERIES.items()):
    s = m16[m16.series_id == sid].set_index("obs_date").value
    ax.plot(s.index, s, lw=0.9); ax.set_ylabel(name, fontsize=8)
    ax.axvline(pd.Timestamp("2023-01-01"), color="k", lw=0.8)
axes[0].set_title("FRED series, 2016-2023 (vertical line: start of validation)", fontsize=10)
fig.tight_layout(); save(fig, "04_fred_series"); plt.show()

## 2. Lead-lag: overnight US moves vs the Nifty

Each series' overnight change (known at 08:30 IST on d) against the Nifty's return on d, its 5-day return
d..d+4, the split of day d into the opening gap (before a 09:15 entry) and the intraday move (after it), and
open(d)→close(d+4), the 5-day move a trade entered at the open actually earns.

In [ ]:
nc, no = NIFTY[["close"]].rename(columns={"close": "N"}), NIFTY[["open"]].rename(columns={"open": "N"})
rn = R.log_returns(nc)["N"]
ngap, nintra = R.gap_intraday(no, nc)
targets = {"Nifty r(d)": rn, "Nifty gap(d)": ngap["N"], "Nifty intraday(d)": nintra["N"],
           "Nifty r(d..d+4)": R.forward_sum(rn.to_frame(), 5)["N"],
           "Nifty open(d)->close(d+4)": R.label_returns(nc, no, None, 5, "open")["N"]}   # what a 09:15 entry earns
rows = []
for period, sel in [("train", DAYS <= TRAIN_END), ("2023", DAYS > TRAIN_END)]:
    for sid in SERIES:
        for tname, t in targets.items():
            res = M.block_bootstrap_corr(chg.loc[sel, sid], t[sel], block=20, n_boot=NBOOT, seed=SEED)
            rows.append({"period": period, "series": sid, "target": tname, **res})
ll = pd.DataFrame(rows)
ll["excludes 0"] = (ll.lo > 0) | (ll.hi < 0)
ll["text"] = ll.apply(lambda r: f"{r['corr']:+.2f} [{r['lo']:+.2f}, {r['hi']:+.2f}]" + (" *" if r["excludes 0"] else ""), axis=1)
display(ll.pivot_table(index=["period", "series"], columns="target", values="text", aggfunc="first")[list(targets)])
print("* = 95% block-bootstrap interval excludes 0; n per cell:", ll.groupby("period").n.median().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(17, 4))
for ax, period in zip(axes, ["train", "2023"]):
    piv = ll[ll.period == period].pivot(index="series", columns="target", values="corr")[list(targets)].loc[list(SERIES)]
    heatmap(ax, piv, f"Correlation with overnight US change, {period}", cmap="RdBu_r", vmin=-0.5, vmax=0.5, fmt="{:+.2f}")
fig.tight_layout(); save(fig, "04_lead_lag"); plt.show()

In [ ]:
# How much of the Nifty's opening gap do overnight US moves explain? (train; days with all five changes)
from numpy.linalg import lstsq
d = pd.concat([chg[list(SERIES)], ngap["N"].rename("gap"), nintra["N"].rename("intra")], axis=1).loc[:TRAIN_END].dropna()
X = np.column_stack([np.ones(len(d)), d[list(SERIES)].to_numpy()])
for target in ["gap", "intra"]:
    beta, *_ = lstsq(X, d[target].to_numpy(), rcond=None)
    resid = d[target].to_numpy() - X @ beta
    print(f"R^2 of Nifty {target} on the five overnight changes (train, n={len(d)}): {1 - resid.var() / d[target].var():.3f}")

## 3. US VIX vs India VIX

In [ ]:
us = vals["VIXCLS"]                                   # US VIX known at 08:30 on d (US close of d-1)
iv_prev = IVIX.shift(1)                               # India VIX close of d-1, also known at 08:30 on d
lv = M.block_bootstrap_corr(us, iv_prev, n_boot=NBOOT, seed=SEED)
ch_india = np.log(IVIX / IVIX.shift(1))               # India VIX change on d (close d-1 -> close d)
cc = M.block_bootstrap_corr(chg["VIXCLS"], ch_india, n_boot=NBOOT, seed=SEED)
cc_prev = M.block_bootstrap_corr(chg["VIXCLS"], ch_india.shift(1), n_boot=NBOOT, seed=SEED)
print(f"levels (both known at 08:30 on d): corr {lv['corr']:.2f} [{lv['lo']:.2f}, {lv['hi']:.2f}], n={lv['n']}")
print(f"US VIX overnight change vs India VIX change on d: {cc['corr']:+.2f} [{cc['lo']:+.2f}, {cc['hi']:+.2f}]")
print(f"  ... vs India VIX change on d-1 (same calendar day, already known): {cc_prev['corr']:+.2f} [{cc_prev['lo']:+.2f}, {cc_prev['hi']:+.2f}]")
ratio = (iv_prev / us).dropna()
print(f"India VIX / US VIX: median {ratio.median():.2f}, 10-90%: {ratio.quantile(0.1):.2f}-{ratio.quantile(0.9):.2f}")

tr = DAYS <= TRAIN_END
bands_in = G.vix_bands(IVIX, START, TRAIN_END, cfg["regime"]["vix_quantiles"])
bands_us = G.vix_bands(us, START, TRAIN_END, cfg["regime"]["vix_quantiles"])
lab_in = G.vix_regime(IVIX, bands_in)                                    # already lagged: close d-1
lab_us = pd.cut(us, [-np.inf, *bands_us, np.inf], labels=["low", "mid", "high"], right=False).astype("object")
both = pd.DataFrame({"India VIX band": lab_in, "US VIX band": lab_us}).dropna()
agree = (both["India VIX band"] == both["US VIX band"])
print(f"band cut-points (train terciles): India {[round(b, 2) for b in bands_in]}, US {[round(b, 2) for b in bands_us]}")
print(f"same band: train {agree[both.index <= TRAIN_END].mean():.1%}, 2023 {agree[both.index > TRAIN_END].mean():.1%}")
display(pd.crosstab(both["India VIX band"], both["US VIX band"], normalize="all").loc[["low", "mid", "high"], ["low", "mid", "high"]]
        .style.format("{:.1%}"))

fig, axes = plt.subplots(1, 3, figsize=(17, 4), gridspec_kw={"width_ratios": [2, 1, 1]})
axes[0].plot(IVIX, lw=0.9, label="India VIX"); axes[0].plot(us, lw=0.9, label="US VIX (as known at 08:30 IST)")
axes[0].axvline(pd.Timestamp("2023-01-01"), color="k", lw=0.8); axes[0].legend(fontsize=8)
axes[0].set_title("India VIX vs US VIX", fontsize=10)
axes[1].scatter(us, iv_prev, s=3, alpha=0.4); axes[1].set_xlabel("US VIX"); axes[1].set_ylabel("India VIX (previous close)")
axes[1].set_title(f"Levels, corr {lv['corr']:.2f}", fontsize=10)
d2 = pd.concat([chg["VIXCLS"], ch_india], axis=1, keys=["us", "in"]).dropna()
axes[2].scatter(d2.us, d2["in"], s=3, alpha=0.4); axes[2].set_xlabel("US VIX overnight log change"); axes[2].set_ylabel("India VIX log change on d")
axes[2].set_title(f"Changes, corr {cc['corr']:+.2f}", fontsize=10)
fig.tight_layout(); save(fig, "04_vix"); plt.show()

In [ ]:
# Volatility: does the US VIX (known at 08:30) add to India VIX for the Nifty's next 5-day volatility?
fvol = R.forward_vol(rn.to_frame(), 5)["N"]                 # std of daily log returns over d..d+4
absintra = nintra["N"].abs()
rows = []
for period, sel in [("train", DAYS <= TRAIN_END), ("2023", DAYS > TRAIN_END)]:
    d = pd.DataFrame({"india": np.log(iv_prev), "us": np.log(us), "dus": chg["VIXCLS"].abs(),
                      "fvol": np.log(fvol), "absintra": absintra})[sel].dropna()
    resid_us = d.us - np.polyval(np.polyfit(d.india, d.us, 1), d.india)          # US VIX not explained by India VIX
    resid_f = d.fvol - np.polyval(np.polyfit(d.india, d.fvol, 1), d.india)       # forward vol not explained by India VIX
    for name, x, y in [("log India VIX vs log next-5d vol", d.india, d.fvol), ("log US VIX vs log next-5d vol", d.us, d.fvol),
                       ("US VIX beyond India VIX vs next-5d vol beyond India VIX", resid_us, resid_f),
                       ("|US VIX overnight change| vs |Nifty intraday(d)|", d.dus, d.absintra)]:
        rows.append({"period": period, "pair": name, **M.block_bootstrap_corr(x, y, n_boot=NBOOT, seed=SEED)})
vv = pd.DataFrame(rows).set_index(["period", "pair"])
display(vv.style.format({"n": "{:,.0f}", "corr": "{:+.2f}", "lo": "{:+.2f}", "hi": "{:+.2f}"}))

## 4. RBI policy decisions

In [ ]:
rbi = pd.read_csv(cfg.manual_dir / "rbi_policy.csv")
print(f"data/manual/rbi_policy.csv: {len(rbi)} rows; columns {list(rbi.columns)}")
if rbi.empty:
    print("BLOCKED on task 4.3: the RBI decisions (about 65 since 2016) have not been entered yet. "
          "Once they are, re-run this cell to measure the Nifty and bank-stock reaction on day d+1 "
          "(decisions are announced around 10:00 IST, so they inform the next trading day).")

## 5. FII/DII flows

In [ ]:
fii = con.execute("SELECT trade_date, category, buy_value_cr, sell_value_cr, net_value_cr, published_at FROM fii_dii").df()
display(fii)
print(f"{len(fii)} rows covering {fii.trade_date.nunique()} day(s). "
      "BLOCKED on tasks 4.1-4.2: NSE's endpoint returns only the latest day, so history grows one day at a time "
      "from the daily job (4.1); FII history must come from NSDL's FPI reports and a DII source is still to be found (4.2). "
      "Nothing in 2016-2023 can be analysed yet. Flows for day d are posted around 20:00 IST, so they inform d+1.")

## 6. News (GDELT)

In [ ]:
news = con.execute("SELECT symbol, published_at, source_domain, title, tone, relevance, match_alias, in_title FROM news").df()
uni = pd.read_csv(cfg.manual_dir / "universe.csv").set_index("symbol")
print(f"news: {len(news)} rows, {news.symbol.nunique()} stocks, published {news.published_at.min()} to {news.published_at.max()}")

def names(sym):
    al = str(uni.loc[sym, "aliases"]) if sym in uni.index and pd.notna(uni.loc[sym, "aliases"]) else ""
    return [a.strip().lower() for a in al.split(";") if a.strip()] + [sym.lower()]

news["company in title"] = [any(a in str(t).lower() for a in names(s)) for s, t in zip(news.symbol, news.title)]
display(news[["symbol", "source_domain", "title", "match_alias", "relevance", "company in title"]])
print(f"rows whose title names the company (alias or symbol): {news['company in title'].sum()} of {len(news)} "
      f"({news['company in title'].mean():.0%})")
print("BLOCKED on tasks 5.1-5.2: only the June 2024 pilot files are filtered, so there is no 2016-2023 news to "
      "measure volume or tone per stock (and the GDELT history route starts in Oct 2022 by design).")

## Findings

The period is 2016–2023; train = 2016–2022. Macro values are aligned on `known_from`: day d sees the latest value public by 08:30 IST. Intervals are 95% moving-block bootstrap intervals (20-day blocks).

1. **The FRED data is clean, and the time rule holds.**
   - Each series has 2,086 rows in 2016–2023, of which 2.6–4.6% have no value (US holidays, about 10 a year).
   - `known_from − obs_date` is exactly 1 day for every row, so values are usable from 05:30 IST the next day.
   - On NSE days the value used is a median 1 day old (at most 4–6 days, after US holidays and long weekends), and a new value arrives on 95.8–97.9% of NSE days.
   - FRED publishes USD/INR, the dollar index and Brent weekly, but their observations are daily market prices. So weekly publishing matters only for the live daily job (task 4.5), not the backtest.
2. **Overnight US moves show up in the Nifty's opening gap, not after it.** Train correlations of each overnight change with the Nifty's gap on d:

   | Series | Correlation with the gap |
   |---|---|
   | US VIX | −0.39 [−0.50, −0.30] |
   | Dollar index | −0.18 |
   | USD/INR | −0.15 |
   | Brent | +0.09 |
   | US 10-year yield | about 0 |

   Together the five explain 16.5% of the gap's variance and 0.3% of the intraday move's (R², train). Correlation with the intraday move (09:15 → close) is within ±0.04 for every series, and none is significant. 2023 looks the same: US VIX vs the gap is −0.46.
3. **No macro move predicts what a trade entered at the open earns.** Correlation with open(d)→close(d+4) is within ±0.06 for all five series, with every interval including 0, in both train and 2023. The close-to-close 5-day return looks linked (US VIX −0.14) only because it includes day d's gap.
4. **US VIX vs India VIX:**
   - Levels correlate 0.80 [0.67, 0.87], and India VIX runs at a median 0.94× US VIX.
   - The US VIX overnight change correlates +0.22 with India VIX's change on d.
   - They fall in the same tercile band on 65% of train days but only 33% of 2023 days, when India VIX sat low while US VIX was mid.
5. **For volatility, India VIX is the input; US VIX adds little.** Correlation with the Nifty's next-5-day volatility (train): India VIX 0.58, US VIX 0.51. US VIX adds only +0.10 [−0.00, +0.20] beyond India VIX. In 2023 both fall to 0.27.
6. **RBI policy is blocked on task 4.3.** `data/manual/rbi_policy.csv` has 0 rows, so the reaction of the Nifty and bank stocks on day d+1 can't be measured yet. Re-run cell 4 once the roughly 65 decisions are entered.
7. **FII/DII flows are blocked on tasks 4.1–4.2.** There are 2 rows (FII −₹944 crore, DII +₹1,924 crore on 29 Oct 2026) and nothing in 2016–2023. History needs the daily job (4.1) and NSDL's FPI reports, plus a DII source (4.2).
8. **News is blocked on tasks 5.1–5.2.** There are only the 14 June 2024 pilot rows (6 stocks). Their precision is poor: only 1 of 14 headlines (7%) names the matched company. Examples are a NASA spacesuit story matched to TRENT and Bollywood stories matched to INFY. Company matching (open item 8) needs fixing before the full GDELT run.

### Which macro inputs are worth passing to the models and agents

- **India VIX:** yes. It's the main volatility input for vol forecasts, regimes and stop width, as in E02.
- **US VIX (overnight change and level):** pass it to the agents as pre-open context, and use it to anticipate the opening gap when setting entry, stop and target levels against the previous close. For the models it's optional: it barely improves the volatility forecast and doesn't predict returns after the open.
- **Dollar index and USD/INR:** the same role, weaker. They explain part of the gap only.
- **Brent and the US 10-year yield:** no measurable link to the Nifty. Keep them as agent context at most, not model features.
- **RBI decisions, FII/DII flows, news:** decide after tasks 4.3, 4.1–4.2 and 5.1–5.2 deliver data.
- **Alignment:** every model or agent reading macro data should go through `macro_asof` (or the task 6.6 point-in-time reader built from it). Then values are used at most as of the previous US close, and the staleness after US holidays (up to 6 days) is explicit.